In [ ]:
from pathlib import Path
import random
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        "PyTorch is not installed. Run: pip install -r requirements.txt"
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

TEXT_COLUMN = "equation"
TEST_SIZE = 0.2
VALID_SIZE_FROM_TRAIN = 0.1
MIN_TOKEN_FREQ = 1
MAX_LEN_PERCENTILE = 95
BATCH_SIZE = 64
EPOCHS = 25
LEARNING_RATE = 1e-3
DROPOUT = 0.25
EMBED_DIM = 96
NUM_FILTERS = 128
PATIENCE = 5

DROP_DUPLICATE_EQUATIONS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def normalize_answer(s: str) -> str:
    if not s:
        return ""

    s = str(s)
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)

    if "=" in s:
        s = s.split("=", 1)[-1]

    s = s.replace("\\left", "").replace("\\right", "")

    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", s)
    s = re.sub(r"e\^\{([^{}]+)\}", r"exp(\1)", s)

    s = re.sub(r"\s+", "", s)
    return s


def normalize_equation(s: str) -> str:
    """Normalize TeX while preserving both sides of the equation for classification."""
    if not s:
        return ""

    s = str(s)
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)
    s = s.replace("\\left", "").replace("\\right", "")
    s = re.sub(r"\\frac\{([^{}]+)\}\{([^{}]+)\}", r"(\1)/(\2)", s)
    s = re.sub(r"\\sqrt\{([^{}]+)\}", r"sqrt(\1)", s)
    s = re.sub(r"e\^\{([^{}]+)\}", r"exp(\1)", s)
    s = re.sub(r"\s+", "", s)
    return s


def tokenize_math(expr: str):
    return re.findall(r"[A-Za-z]+|\d+|\+|\-|\*|\/|\(|\)|=|\^|\{|\}|_|'", expr)


def preprocess_equation(equation: str) -> tuple[str, list[str], str]:
    normalized = normalize_equation(equation)
    tokens = tokenize_math(normalized)
    token_text = " ".join(tokens)
    return normalized, tokens, token_text


example = r"y^{\prime}=\frac{x^{2}}{\sqrt{1-x^{2}}}+e^{2x}"
normalize_equation(example), tokenize_math(normalize_equation(example))[:30]

In [ ]:
combined = pd.read_excel(combined_equations.xlsx)
label_names = sorted(combined["label"].unique())
label_to_id = {label: idx for idx, label in enumerate(label_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
combined["label_id"] = combined["label"].map(label_to_id)

train_df, test_df = train_test_split(
    combined,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=combined["label_id"],
)

train_df, valid_df = train_test_split(
    train_df,
    test_size=VALID_SIZE_FROM_TRAIN,
    random_state=SEED,
    stratify=train_df["label_id"],
)

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict())
print("Valid:", valid_df.shape, valid_df["label"].value_counts().to_dict())
print("Test :", test_df.shape, test_df["label"].value_counts().to_dict())

split_path = DATA_DIR / "combined_equations_with_split.csv"
split_df = combined.copy()
split_df["split"] = "unused"
split_df.loc[train_df.index, "split"] = "train"
split_df.loc[valid_df.index, "split"] = "valid"
split_df.loc[test_df.index, "split"] = "test"
split_df.to_csv(split_path, index=False, encoding="utf-8-sig")
print("Saved split file:", split_path)

In [ ]:
PAD = "<PAD>"
UNK = "<UNK>"

def build_vocab(token_texts: pd.Series, min_freq: int = 1) -> dict[str, int]:
    counter = Counter()
    for text in token_texts:
        counter.update(str(text).split())

    vocab = {PAD: 0, UNK: 1}
    for token, freq in counter.most_common():
        if freq >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(train_df["token_text"], min_freq=MIN_TOKEN_FREQ)
lengths = train_df["token_count"].to_numpy()
max_len = int(np.percentile(lengths, MAX_LEN_PERCENTILE))
max_len = max(16, min(max_len, int(lengths.max())))

print("Vocab size:", len(vocab))
print("Max sequence length:", max_len)


def encode_tokens(token_text: str, vocab: dict[str, int], max_len: int) -> list[int]:
    ids = [vocab.get(token, vocab[UNK]) for token in str(token_text).split()]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [vocab[PAD]] * (max_len - len(ids))
    return ids


class EquationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: dict[str, int], max_len: int):
        self.x = torch.tensor(
            [encode_tokens(text, vocab, max_len) for text in df["token_text"]],
            dtype=torch.long,
        )
        self.y = torch.tensor(df["label_id"].to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_loader = DataLoader(EquationDataset(train_df, vocab, max_len), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(EquationDataset(valid_df, vocab, max_len), batch_size=BATCH_SIZE)
test_loader = DataLoader(EquationDataset(test_df, vocab, max_len), batch_size=BATCH_SIZE)

In [ ]:
class MathTextCNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_classes: int,
        embed_dim: int = 96,
        num_filters: int = 128,
        kernel_sizes: tuple[int, ...] = (3, 5, 7),
        dropout: float = 0.25,
        padding_idx: int = 0,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        self.convs = nn.ModuleList(
            [nn.Conv1d(embed_dim, num_filters, kernel_size=k, padding=k // 2) for k in kernel_sizes]
        )
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        embedded = self.embedding(x).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            features = self.activation(conv(embedded))
            pooled.append(torch.amax(features, dim=-1))
        features = torch.cat(pooled, dim=1)
        features = self.dropout(features)
        return self.classifier(features)


model = MathTextCNN(
    vocab_size=len(vocab),
    num_classes=len(label_names),
    embed_dim=EMBED_DIM,
    num_filters=NUM_FILTERS,
    dropout=DROPOUT,
).to(device)

class_counts = np.bincount(train_df["label_id"].to_numpy(), minlength=len(label_names))
class_weights = class_counts.sum() / np.maximum(class_counts, 1)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
model

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    all_true = []
    all_pred = []

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
                optimizer.step()

        total_loss += loss.item() * len(y)
        preds = logits.argmax(dim=1)
        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_true, all_pred, average="macro")
    accuracy = np.mean(np.array(all_true) == np.array(all_pred))
    return avg_loss, macro_f1, accuracy

best_valid_f1 = -1.0
best_state = None
bad_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_f1, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    valid_loss, valid_f1, valid_acc = run_epoch(model, valid_loader, criterion)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_f1": train_f1,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_f1": valid_f1,
        "valid_acc": valid_acc,
    })

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} f1 {train_f1:.4f} acc {train_acc:.4f} | "
        f"valid loss {valid_loss:.4f} f1 {valid_f1:.4f} acc {valid_acc:.4f}"
    )

    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping after {epoch} epochs")
            break

if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history)